# Notebook 02 — Feature Engineering

**Project:** Customer Review Prediction — Olist Brazilian E-Commerce  

Alright, now for the fun part! I'm going to take that analytical dataset I just built and engineer some smart features for the machine learning model. 

---

## What I'm Doing Here

I want to transform the raw order data into strong **predictive features**. Here's what I'm going to build:

| Feature Group | Features I'm Engineering |
|---|---|
| Time | purchase_month, purchase_weekday, purchase_hour |
| Order | total_order_value, freight_value, freight_ratio, number_of_items, seller_count |
| Product | dominant_product_category, number_of_categories, total_product_weight_g, total_product_volume_cm3 |
| Geographic | customer_state, seller_state, same_state |
| Payment | payment_type, payment_installments |
| Customer History | previous_order_count, previous_total_spent, previous_average_review |
| Seller History | seller_average_review, seller_total_orders, seller_late_delivery_rate |

## The Golden Rule: No Data Leakage!

When I build the historical features (like a seller's average review score), I have to be incredibly careful. I can ONLY use information from orders that happened **before** the current order. If I accidentally include the current order's review to predict the current order, that's data leakage and it ruins the model! I'll make sure to handle this properly.

## 1. Imports and Configuration
Getting my tools ready.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

print('Imports complete.')

Imports complete.


In [2]:
# ---------------------------------------------------------------------------
# File paths — relative to the notebooks/ directory
# ---------------------------------------------------------------------------
ANALYTICAL_DATA_DIR = Path('../data/analytical')
CLEANED_DATA_DIR    = Path('../../cleaned')

ANALYTICAL_DATASET_PATH = ANALYTICAL_DATA_DIR / 'analytical_dataset.csv'
FEATURE_DATASET_PATH    = ANALYTICAL_DATA_DIR / 'feature_dataset.csv'

print(f'Reading from : {ANALYTICAL_DATASET_PATH.resolve()}')
print(f'Writing to   : {FEATURE_DATASET_PATH.resolve()}')

Reading from : C:\Users\VishalReddyK\OneDrive\Documents\semester 2\statiscal machine learning\main project\customer_review_prediction\data\analytical\analytical_dataset.csv
Writing to   : C:\Users\VishalReddyK\OneDrive\Documents\semester 2\statiscal machine learning\main project\customer_review_prediction\data\analytical\feature_dataset.csv


## 2. Load Data
Let's grab the dataset I saved in notebook 1.

In [3]:
# Load the analytical dataset produced in Notebook 01
analytical_df = pd.read_csv(ANALYTICAL_DATASET_PATH)

print(f'Analytical dataset loaded: {analytical_df.shape}')

# We also need the raw orders and reviews data to compute SELLER historical
# features over ALL orders (not just the binary-review subset in analytical_df).
# This ensures seller history is as complete as possible.
orders_df      = pd.read_csv(CLEANED_DATA_DIR / 'olist_orders_dataset.csv')
order_items_df = pd.read_csv(CLEANED_DATA_DIR / 'olist_order_items_dataset.csv')
reviews_df     = pd.read_csv(CLEANED_DATA_DIR / 'olist_order_reviews_dataset.csv')

print(f'Supporting datasets loaded.')
print(f'  orders      : {orders_df.shape}')
print(f'  order_items : {order_items_df.shape}')
print(f'  reviews     : {reviews_df.shape}')

Analytical dataset loaded: (90540, 25)


Supporting datasets loaded.
  orders      : (99441, 8)
  order_items : (112650, 7)
  reviews     : (98673, 7)


In [4]:
# Parse all timestamp columns to datetime objects.
# We need datetime for time-based feature extraction and for sorting orders
# chronologically when computing historical features.

timestamp_columns_in_analytical = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in timestamp_columns_in_analytical:
    if col in analytical_df.columns:
        analytical_df[col] = pd.to_datetime(analytical_df[col])

# Parse timestamps in supporting tables too
orders_df['order_purchase_timestamp']        = pd.to_datetime(orders_df['order_purchase_timestamp'])
orders_df['order_delivered_customer_date']   = pd.to_datetime(orders_df['order_delivered_customer_date'])
orders_df['order_estimated_delivery_date']   = pd.to_datetime(orders_df['order_estimated_delivery_date'])

print('Timestamps parsed successfully.')
print(f"Date range: {analytical_df['order_purchase_timestamp'].min().date()} to "
      f"{analytical_df['order_purchase_timestamp'].max().date()}")

Timestamps parsed successfully.
Date range: 2016-09-04 to 2018-10-17


---

## 3. Time Features

People definitely shop differently at 2 AM on a Sunday compared to 3 PM on a Tuesday. Also, holiday rushes can impact shipping speeds. So, I'm going to extract some time-based features to see if they affect reviews.

| Feature | What it tells me |
|---|---|
| `purchase_month` | Seasonal trends (like the Christmas rush) |
| `purchase_weekday` | Weekend vs weekday shopping |
| `purchase_hour` | Time-of-day habits |

In [5]:
def extract_time_features(dataframe: pd.DataFrame,
                          timestamp_column: str) -> pd.DataFrame:
    """
    Extract calendar and time-of-day features from a datetime column.

    Parameters
    ----------
    dataframe : pd.DataFrame
        The input DataFrame (will not be modified; a copy is returned).
    timestamp_column : str
        Name of the datetime column to extract features from.

    Returns
    -------
    pd.DataFrame
        DataFrame with three new columns:
        purchase_month, purchase_weekday (0=Monday … 6=Sunday), purchase_hour.
    """
    result = dataframe.copy()

    # Month (1–12): captures seasonal purchasing patterns
    result['purchase_month'] = result[timestamp_column].dt.month

    # Day of week (0=Monday, 6=Sunday): distinguishes weekday from weekend shopping
    result['purchase_weekday'] = result[timestamp_column].dt.dayofweek

    # Hour (0–23): captures time-of-day effects
    result['purchase_hour'] = result[timestamp_column].dt.hour

    return result


analytical_df = extract_time_features(analytical_df, 'order_purchase_timestamp')

print('Time features added:')
print(f"  purchase_month   : {analytical_df['purchase_month'].unique()  }")
print(f"  purchase_weekday : range 0–6 (Mon=0, Sun=6)")
print(f"  purchase_hour    : range 0–23")

Time features added:
  purchase_month   : [10  7  8 11  2  4  5  1  6  3 12  9]
  purchase_weekday : range 0–6 (Mon=0, Sun=6)
  purchase_hour    : range 0–23


---

## 4. Order Features

I have a feeling that order cost and complexity matter. If I pay a ton for shipping relative to the item price, I'm going to be annoyed. Also, orders with lots of items might get messed up more easily.

| Feature | What it is |
|---|---|
| `total_order_value` | Price of the items (no shipping) |
| `freight_value` | How much shipping cost |
| `freight_ratio` | What percentage of the total bill was just shipping? |
| `number_of_items` | How many things are in the box? |
| `seller_count` | How many sellers are fulfilling this single order? |

In [6]:
# Rename total_price → total_order_value for clarity
analytical_df = analytical_df.rename(columns={'total_price': 'total_order_value'})

# Freight ratio: freight cost as a fraction of the total order amount
# (total_order_value + freight_value)
# We use a safe denominator: if both are zero, the ratio is 0
total_order_amount = analytical_df['total_order_value'] + analytical_df['total_freight_value']
analytical_df['freight_ratio'] = np.where(
    total_order_amount > 0,
    analytical_df['total_freight_value'] / total_order_amount,
    0.0
)

# Rename for consistency with the feature specification
analytical_df = analytical_df.rename(columns={'total_freight_value': 'freight_value'})

print('Order features ready:')
print(analytical_df[['total_order_value', 'freight_value', 'freight_ratio',
                      'number_of_items', 'seller_count']].describe())

Order features ready:
       total_order_value  freight_value  freight_ratio  number_of_items  \
count        89,837.0000    89,837.0000    90,540.0000      89,837.0000   
mean            138.4244        22.7382         0.2064           1.1391   
std             212.7872        21.5610         0.1264           0.5304   
min               0.8500         0.0000         0.0000           1.0000   
25%              45.9500        13.7900         0.1144           1.0000   
50%              86.9900        17.1400         0.1819           1.0000   
75%             149.9000        23.9500         0.2740           1.0000   
max          13,440.0000     1,794.9600         0.9555          21.0000   

       seller_count  
count   89,837.0000  
mean         1.0129  
std          0.1194  
min          1.0000  
25%          1.0000  
50%          1.0000  
75%          1.0000  
max          5.0000  


---

## 5. Geographic Features

Brazil is huge. Shipping stuff across the country takes time and money. I'm going to create a `same_state` flag to track if the buyer and seller are in the same state, which usually means faster delivery.

In [7]:
def compute_geographic_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Add the same_state binary feature to the DataFrame.

    same_state = 1 if the customer and the primary seller are in the same
    Brazilian state, 0 otherwise.  This is a proxy for proximity and
    likely delivery speed.

    Parameters
    ----------
    dataframe : pd.DataFrame
        Must contain 'customer_state' and 'seller_state' columns.

    Returns
    -------
    pd.DataFrame
        DataFrame with an additional 'same_state' integer column.
    """
    result = dataframe.copy()

    # same_state is 1 only when both states are non-null and equal
    result['same_state'] = (
        result['customer_state'].notna()
        & result['seller_state'].notna()
        & (result['customer_state'] == result['seller_state'])
    ).astype(int)

    return result


analytical_df = compute_geographic_features(analytical_df)

same_state_pct = analytical_df['same_state'].mean() * 100
print(f'Same-state orders: {analytical_df["same_state"].sum():,} ({same_state_pct:.1f}%)')
print(f'Cross-state orders: {(analytical_df["same_state"] == 0).sum():,} ({100 - same_state_pct:.1f}%)')

Same-state orders: 32,524 (35.9%)
Cross-state orders: 58,016 (64.1%)


---

## 6. Customer Historical Features

Does a repeat customer have different expectations than a first-time buyer? Let's find out! I'm going to calculate some history for each customer.

**How I'm avoiding leakage:** I'm sorting the orders chronologically and using `cumcount()` and a shifted `cumsum()`. This guarantees that when I look at "Order #3", I'm only using stats from Order #1 and Order #2. It's a neat trick!

| Feature | What it is |
|---|---|
| `previous_order_count` | How many times have they bought from us before? |
| `previous_total_spent` | How much money have they spent with us in the past? |
| `previous_average_review` | Are they generally a harsh grader or an easy grader? |

In [8]:
def compute_customer_historical_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Compute historical purchase statistics for each customer.

    For each order, the historical features reflect ONLY the orders placed
    by the same customer BEFORE the current order's purchase timestamp.
    This prevents data leakage from future orders into the current one.

    The approach:
    1. Sort by (customer_unique_id, order_purchase_timestamp) so orders are
       processed chronologically within each customer group.
    2. Use cumcount() for the prior order count (0-indexed = number of orders before).
    3. Use cumsum().shift(1) for running totals, so the current row is excluded.

    Parameters
    ----------
    dataframe : pd.DataFrame
        Must contain: customer_unique_id, order_purchase_timestamp,
        total_order_value, freight_value, review_score.

    Returns
    -------
    pd.DataFrame
        Original DataFrame with three new columns:
        previous_order_count, previous_total_spent, previous_average_review.
    """
    # Sort chronologically within each customer group.
    # We sort by the customer_unique_id first, then by timestamp,
    # so that all of a customer's orders appear in the right order.
    sorted_df = dataframe.sort_values(
        by=['customer_unique_id', 'order_purchase_timestamp']
    ).copy()

    # -----------------------------------------------------------------------
    # previous_order_count
    # cumcount() returns the within-group position (0-indexed).
    # The first order gets 0, the second gets 1, etc.
    # This is exactly the number of orders BEFORE the current one.
    # -----------------------------------------------------------------------
    sorted_df['previous_order_count'] = (
        sorted_df
        .groupby('customer_unique_id')
        .cumcount()   # 0 for first order, 1 for second, etc.
    )

    # -----------------------------------------------------------------------
    # previous_total_spent
    # Total amount spent = sum of (item price + freight) on all prior orders.
    # cumsum().shift(1) gives the running total BEFORE the current order.
    # The first order in each group gets NaN (no prior orders), filled with 0.
    # -----------------------------------------------------------------------
    total_order_amount = sorted_df['total_order_value'] + sorted_df['freight_value']

    sorted_df['previous_total_spent'] = (
        total_order_amount
        .groupby(sorted_df['customer_unique_id'])
        .cumsum()
        .shift(1)        # exclude the current order from its own history
        .fillna(0.0)     # first-time customers have no prior spending
    )

    # -----------------------------------------------------------------------
    # previous_average_review
    # Average of all review scores the customer gave on prior orders.
    # We compute cumulative sum and count of review scores, then shift both
    # by 1 to exclude the current order.
    # First-time customers receive NaN (no review history).
    # -----------------------------------------------------------------------
    cumulative_review_sum = (
        sorted_df['review_score']
        .groupby(sorted_df['customer_unique_id'])
        .cumsum()
        .shift(1)   # exclude current order
    )

    # previous_order_count already gives us the count of prior orders
    prior_order_count_for_review = sorted_df['previous_order_count']

    sorted_df['previous_average_review'] = np.where(
        prior_order_count_for_review > 0,
        cumulative_review_sum / prior_order_count_for_review,
        np.nan    # NaN for first-time customers — handled later in the model pipeline
    )

    return sorted_df


analytical_df = compute_customer_historical_features(analytical_df)

print('Customer historical features computed.')
print()
print('previous_order_count distribution:')
print(analytical_df['previous_order_count'].value_counts().head(10))
print()
print('previous_average_review statistics:')
print(analytical_df['previous_average_review'].describe())

Customer historical features computed.

previous_order_count distribution:
previous_order_count
0    87676
1     2575
2      208
3       44
4       16
5        7
6        4
7        1
8        1
9        1
Name: count, dtype: int64

previous_average_review statistics:
count   2,864.0000
mean        4.2531
std         1.3263
min         1.0000
25%         4.0000
50%         5.0000
75%         5.0000
max         5.0000
Name: previous_average_review, dtype: float64


---

## 7. Seller Historical Features

This one is huge. A seller with a terrible track record is probably going to disappoint the next customer too. 

**How I'm doing it:** Just like with customers, I'll build a chronological timeline for each seller and calculate their running stats. I have to do this on the *entire* dataset (not just the subset with reviews) to get accurate counts, and then I'll join it back.

| Feature | What it is |
|---|---|
| `seller_average_review` | What was this seller's average rating *before* this order? |
| `seller_total_orders` | How experienced is this seller? |
| `seller_late_delivery_rate` | How often do they ship late? |

In [9]:
def build_seller_order_timeline(orders_df: pd.DataFrame,
                                 order_items_df: pd.DataFrame,
                                 reviews_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a timeline of all (seller, order) pairs with timestamps,
    review scores, and late-delivery flags.

    This uses ALL orders (not just the binary-review subset) so that
    seller historical statistics are as complete as possible.

    Parameters
    ----------
    orders_df : pd.DataFrame
        Full orders table with timestamps.
    order_items_df : pd.DataFrame
        Full order items table with seller_id per item.
    reviews_df : pd.DataFrame
        Full reviews table with review_score per order.

    Returns
    -------
    pd.DataFrame
        One row per (seller_id, order_id) with columns:
        seller_id, order_id, order_purchase_timestamp,
        review_score, is_late_delivery.
    """
    # Get distinct (order_id, seller_id) pairs
    seller_order_pairs = order_items_df[['order_id', 'seller_id']].drop_duplicates()

    # Compute late-delivery flag from orders:
    # An order is late if the actual delivery date exceeded the estimated date
    orders_with_late_flag = orders_df[[
        'order_id',
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]].copy()

    orders_with_late_flag['is_late_delivery'] = (
        orders_with_late_flag['order_delivered_customer_date']
        > orders_with_late_flag['order_estimated_delivery_date']
    ).astype(float)  # float so NaN propagates when dates are missing

    # Deduplicate reviews (keep one review per order)
    reviews_dedup = reviews_df[['order_id', 'review_score']].drop_duplicates('order_id')

    # Join everything together
    seller_timeline = (
        seller_order_pairs
        .merge(
            orders_with_late_flag[['order_id', 'order_purchase_timestamp', 'is_late_delivery']],
            on='order_id',
            how='left'
        )
        .merge(
            reviews_dedup,
            on='order_id',
            how='left'
        )
    )

    return seller_timeline


seller_timeline = build_seller_order_timeline(orders_df, order_items_df, reviews_df)

print(f'Seller-order timeline built: {seller_timeline.shape}')
print(f'Unique sellers  : {seller_timeline["seller_id"].nunique():,}')
print(f'Unique orders   : {seller_timeline["order_id"].nunique():,}')

Seller-order timeline built: (100010, 5)
Unique sellers  : 3,095
Unique orders   : 98,666


In [10]:
def compute_seller_historical_features(seller_timeline: pd.DataFrame) -> pd.DataFrame:
    """
    Compute historical seller performance metrics for each (seller, order) pair.

    For each order, the metrics reflect only the seller's performance on
    orders that were placed BEFORE the current order (no leakage).

    The approach mirrors the customer historical feature computation:
    sort chronologically, use cumcount() for prior order count, and
    shifted cumsum() for running totals.

    Parameters
    ----------
    seller_timeline : pd.DataFrame
        Output of build_seller_order_timeline().

    Returns
    -------
    pd.DataFrame
        DataFrame with seller_id, order_id, seller_average_review,
        seller_total_orders, seller_late_delivery_rate.
    """
    # Sort each seller's orders chronologically
    timeline_sorted = seller_timeline.sort_values(
        by=['seller_id', 'order_purchase_timestamp']
    ).copy()

    # Number of orders this seller has completed BEFORE the current one
    timeline_sorted['seller_total_orders'] = (
        timeline_sorted.groupby('seller_id').cumcount()
    )

    # Cumulative sum of review scores up to (but not including) the current order
    cumulative_review_sum = (
        timeline_sorted['review_score']
        .groupby(timeline_sorted['seller_id'])
        .cumsum()
        .shift(1)
    )

    # Cumulative count of reviews (only where a review score exists)
    review_exists = timeline_sorted['review_score'].notna().astype(float)
    cumulative_review_count = (
        review_exists
        .groupby(timeline_sorted['seller_id'])
        .cumsum()
        .shift(1)
    )

    # Seller average review = cumulative sum / cumulative count
    # NaN when the seller has no prior reviews
    timeline_sorted['seller_average_review'] = np.where(
        (cumulative_review_count > 0) & cumulative_review_count.notna(),
        cumulative_review_sum / cumulative_review_count,
        np.nan
    )

    # Cumulative sum of late deliveries up to (but not including) the current order
    cumulative_late_sum = (
        timeline_sorted['is_late_delivery']
        .groupby(timeline_sorted['seller_id'])
        .cumsum()
        .shift(1)
    )

    # Cumulative count of deliveries with a known on-time/late status
    delivery_known = timeline_sorted['is_late_delivery'].notna().astype(float)
    cumulative_delivery_count = (
        delivery_known
        .groupby(timeline_sorted['seller_id'])
        .cumsum()
        .shift(1)
    )

    # Seller late delivery rate
    timeline_sorted['seller_late_delivery_rate'] = np.where(
        (cumulative_delivery_count > 0) & cumulative_delivery_count.notna(),
        cumulative_late_sum / cumulative_delivery_count,
        np.nan
    )

    seller_features = timeline_sorted[[
        'seller_id',
        'order_id',
        'seller_average_review',
        'seller_total_orders',
        'seller_late_delivery_rate'
    ]]

    return seller_features


seller_historical_features = compute_seller_historical_features(seller_timeline)

print('Seller historical features computed.')
print()
print(seller_historical_features[['seller_average_review',
                                   'seller_total_orders',
                                   'seller_late_delivery_rate']].describe())

Seller historical features computed.

       seller_average_review  seller_total_orders  seller_late_delivery_rate
count            99,246.0000         100,010.0000               100,009.0000
mean                  4.0997             186.6507                     0.0678
std                   0.4786             303.0752                     0.0857
min                   1.0000               0.0000                     0.0000
25%                   3.9434              15.0000                     0.0119
50%                   4.1340              60.0000                     0.0513
75%                   4.3333             202.0000                     0.0926
max                   5.0000           1,853.0000                     1.0000


In [11]:
# Join seller historical features to the analytical dataset.
# We match on BOTH order_id AND primary_seller_id so that we get
# the correct historical stats for the specific seller in each order.

analytical_df = analytical_df.merge(
    seller_historical_features.rename(columns={'seller_id': 'primary_seller_id'}),
    on=['order_id', 'primary_seller_id'],
    how='left'
)

print(f'Seller historical features joined. Dataset shape: {analytical_df.shape}')
print()
print('Seller features missing value summary:')
for col in ['seller_average_review', 'seller_total_orders', 'seller_late_delivery_rate']:
    missing = analytical_df[col].isna().sum()
    pct = missing / len(analytical_df) * 100
    print(f'  {col:<30}: {missing:>6,} missing ({pct:.1f}%)')

Seller historical features joined. Dataset shape: (90540, 36)

Seller features missing value summary:
  seller_average_review         :  1,380 missing (1.5%)
  seller_total_orders           :    703 missing (0.8%)
  seller_late_delivery_rate     :    704 missing (0.8%)


---

## 8. Product Features

Heavy, bulky stuff is hard to ship and breaks easily. That's bound to cause some 1-star reviews. I actually already calculated the total weight and volume back in Notebook 1, so here I'm just verifying everything looks good.

In [12]:
product_feature_columns = [
    'dominant_product_category',
    'number_of_categories',
    'total_product_weight_g',
    'total_product_volume_cm3'
]

print('Product feature summary:')
print(analytical_df[product_feature_columns].describe())

print(f'\nUnique dominant categories: {analytical_df["dominant_product_category"].nunique()}')
print(f'\nTop 10 dominant categories:')
print(analytical_df['dominant_product_category'].value_counts().head(10))

Product feature summary:
       number_of_categories  total_product_weight_g  total_product_volume_cm3
count           89,837.0000             89,837.0000               89,837.0000
mean                 1.0079              2,372.1053               17,286.4237
std                  0.0903              4,740.3232               30,201.0628
min                  1.0000                  0.0000                    0.0000
25%                  1.0000                300.0000                2,964.0000
50%                  1.0000                750.0000                7,200.0000
75%                  1.0000              2,050.0000               19,800.0000
max                  3.0000            184,400.0000            1,476,000.0000

Unique dominant categories: 74

Top 10 dominant categories:
dominant_product_category
bed_bath_table           8317
health_beauty            8055
sports_leisure           7082
computers_accessories    6091
furniture_decor          5737
housewares               5290
watche

---

## 9. Payment Features

How people pay tells you a lot about them. Are they splitting a $20 purchase into 10 installments? That might mean they are super budget-conscious and might be less forgiving if things go wrong. I already pulled this data in Notebook 1.

In [13]:
print('Payment type distribution:')
print(analytical_df['payment_type'].value_counts())

print(f'\nPayment installments distribution:')
print(analytical_df['payment_installments'].describe())

Payment type distribution:
payment_type
credit_card    68349
boleto         17952
voucher         2832
debit_card      1404
not_defined        2
Name: count, dtype: int64

Payment installments distribution:
count   90,539.0000
mean         2.9223
std          2.7157
min          0.0000
25%          1.0000
50%          2.0000
75%          4.0000
max         24.0000
Name: payment_installments, dtype: float64


---

## 10. Build and Save the Feature Dataset

Awesome, the features are ready! I'll just select the columns I need (plus the `order_id` so I don't lose track of things) and save it. Time to do some EDA!

In [14]:
# Define the final set of features for machine learning
# Each feature group is listed separately for clarity and maintainability

IDENTIFIER_COLUMNS = ['order_id']

TARGET_COLUMN = ['positive_review']

TIME_FEATURES = [
    'purchase_month',
    'purchase_weekday',
    'purchase_hour'
]

ORDER_FEATURES = [
    'total_order_value',
    'freight_value',
    'freight_ratio',
    'number_of_items',
    'seller_count'
]

PRODUCT_FEATURES = [
    'dominant_product_category',
    'number_of_categories',
    'total_product_weight_g',
    'total_product_volume_cm3'
]

GEOGRAPHIC_FEATURES = [
    'customer_state',
    'seller_state',
    'same_state'
]

PAYMENT_FEATURES = [
    'payment_type',
    'payment_installments'
]

CUSTOMER_HISTORICAL_FEATURES = [
    'previous_order_count',
    'previous_total_spent',
    'previous_average_review'
]

SELLER_HISTORICAL_FEATURES = [
    'seller_average_review',
    'seller_total_orders',
    'seller_late_delivery_rate'
]

# Combine all feature groups into a single list
ALL_FEATURE_COLUMNS = (
    TIME_FEATURES
    + ORDER_FEATURES
    + PRODUCT_FEATURES
    + GEOGRAPHIC_FEATURES
    + PAYMENT_FEATURES
    + CUSTOMER_HISTORICAL_FEATURES
    + SELLER_HISTORICAL_FEATURES
)

# Build the feature dataset
columns_to_keep = IDENTIFIER_COLUMNS + ALL_FEATURE_COLUMNS + TARGET_COLUMN
feature_dataset = analytical_df[columns_to_keep].copy()

print(f'Feature dataset shape: {feature_dataset.shape}')
print(f'Total features        : {len(ALL_FEATURE_COLUMNS)}')
print(f'\nFeature columns: {ALL_FEATURE_COLUMNS}')

Feature dataset shape: (90540, 25)
Total features        : 23

Feature columns: ['purchase_month', 'purchase_weekday', 'purchase_hour', 'total_order_value', 'freight_value', 'freight_ratio', 'number_of_items', 'seller_count', 'dominant_product_category', 'number_of_categories', 'total_product_weight_g', 'total_product_volume_cm3', 'customer_state', 'seller_state', 'same_state', 'payment_type', 'payment_installments', 'previous_order_count', 'previous_total_spent', 'previous_average_review', 'seller_average_review', 'seller_total_orders', 'seller_late_delivery_rate']


In [15]:
# Missing value summary for the feature dataset
missing_counts = feature_dataset.isnull().sum()
missing_pcts = (missing_counts / len(feature_dataset) * 100).round(2)

missing_summary = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct': missing_pcts
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

print('Missing Values in Feature Dataset:')
if missing_summary.empty:
    print('  None found.')
else:
    print(missing_summary.to_string())

print()
print('Note: Missing historical features (NaN) reflect first-time customers or')
print('new sellers with no prior orders.  These will be handled by the')
print('imputation step in the ML pipeline (Notebook 04).')

Missing Values in Feature Dataset:
                           missing_count  missing_pct
previous_average_review            87676      96.8400
seller_average_review               1380       1.5200
total_order_value                    703       0.7800
seller_count                         703       0.7800
dominant_product_category            703       0.7800
freight_value                        703       0.7800
number_of_items                      703       0.7800
total_product_weight_g               703       0.7800
number_of_categories                 703       0.7800
total_product_volume_cm3             703       0.7800
seller_state                         703       0.7800
seller_total_orders                  703       0.7800
seller_late_delivery_rate            704       0.7800
payment_installments                   1       0.0000
payment_type                           1       0.0000

Note: Missing historical features (NaN) reflect first-time customers or
new sellers with no prior or

In [16]:
# Save the feature dataset to disk
feature_dataset.to_csv(FEATURE_DATASET_PATH, index=False)

print(f'Feature dataset saved to:')
print(f'  {FEATURE_DATASET_PATH.resolve()}')
print(f'\nRows saved : {len(feature_dataset):,}')
print(f'Columns    : {len(feature_dataset.columns)}')
print()
print('Notebook 02 complete.  Proceed to notebooks/03_eda_hypothesis_testing.ipynb')

Feature dataset saved to:
  C:\Users\VishalReddyK\OneDrive\Documents\semester 2\statiscal machine learning\main project\customer_review_prediction\data\analytical\feature_dataset.csv

Rows saved : 90,540
Columns    : 25

Notebook 02 complete.  Proceed to notebooks/03_eda_hypothesis_testing.ipynb
